<a href="https://colab.research.google.com/github/camille2019/Women-Capital-Trial-Analysis/blob/main/propensity_matching_psm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install psmpy


In [2]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

from psmpy import PsmPy
from psmpy.functions import cohenD
from psmpy.plotting import *

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
save_path = '/content/drive/MyDrive/Capital_trials/'
full_meta = pd.read_csv(save_path + 'full_crime_meta_.csv')
ohe_meta = pd.read_csv('/content/drive/MyDrive/Capital_trials/ohe_metadata_only.csv')

In [25]:
def add_propensity(propensity_columns, control_column, treatment, df_encoded, df_meta, matched_group_name):

  X = df_encoded[propensity_columns]
  T = df_encoded[control_column]


  psm_df =  X.copy()
  psm_df[treatment] = T.astype(int)

  psm_df = psm_df.reset_index(drop=True)
  psm_df["row_id"] = psm_df.index

  psm = PsmPy(psm_df, treatment=treatment, indx="row_id", exclude=[])
  psm.logistic_ps(balance=True)
  pred = psm.predicted_data

  df_meta = df_meta.reset_index(drop=True)

  scores_col_name = matched_group_name + '_prop_scores'

  df_meta[scores_col_name] = pred["propensity_score"].values

  psm.kdtree_matched(matcher="propensity_score", replacement=False, caliper=None)
  matched_df = psm.df_matched
  matched_ids = set(psm.df_matched["row_id"].unique())
  df_meta = df_meta.reset_index(drop=True)

  binary_col_name = 'psmpropensity_' + matched_group_name

  df_meta[binary_col_name] = df_meta.index.isin(matched_ids)

  return df_meta

In [10]:

state_race_crime_cols = ['State_AL', 'State_AZ', 'State_CA','State_FL', 'State_GA',
 'State_LA', 'State_MS', 'State_NC', 'State_OH',
 'State_OK', 'State_PA', 'State_SC', 'State_TN', 'State_TX',
  'Race/Ethnicity_Asian', 'Race/Ethnicity_Black', 'Race/Ethnicity_Latino',
   'Race/Ethnicity_Latinx', 'Race/Ethnicity_Native American',
'Race/Ethnicity_Native American,White,Black', 'Race/Ethnicity_White',
                'Continuing threat to society', 'Disregard for human life', 'Manner - mayhem',
                'Multiple - harm to multiple people', 'Retaliation', 'Victim - old or disabled',
                'Weapon - arson', 'Weapon - deadly weapon', 'aggravated assault', 'attempted rape',
                'burglary/robbery', 'child victim', 'created public risk', 'deliberate and premeditated',
                'done while committing multiplie felonies', 'especially cruel', 'fiancial gain', 'financial gain',
                'financial gian', 'gang murder', 'intentional infliction of great bodily injury', 'interfere with prosecution/arrest',
                'kidnapping', 'larceny', 'lewd acts with a child', 'lying in wait', 'multiple murders',
                'murdered an officer in the line of duty', 'personal use of a handgun', 'poison', 'prior conviction',
                'prior offense', 'prior offenses', 'rape', 'robbery',
                'sexual assult', 'shooting at an occupied vehicle', 'sodomy', 'torture', 'unlawful possession of a firearm']



In [9]:
race_crime_cols = [ 'Race/Ethnicity_Asian', 'Race/Ethnicity_Black', 'Race/Ethnicity_Latino',
   'Race/Ethnicity_Latinx', 'Race/Ethnicity_Native American',
'Race/Ethnicity_Native American,White,Black', 'Race/Ethnicity_White',
                'Continuing threat to society', 'Disregard for human life', 'Manner - mayhem',
                'Multiple - harm to multiple people', 'Retaliation', 'Victim - old or disabled',
                'Weapon - arson', 'Weapon - deadly weapon', 'aggravated assault', 'attempted rape',
                'burglary/robbery', 'child victim', 'created public risk', 'deliberate and premeditated',
                'done while committing multiplie felonies', 'especially cruel', 'fiancial gain', 'financial gain',
                'financial gian', 'gang murder', 'intentional infliction of great bodily injury', 'interfere with prosecution/arrest',
                'kidnapping', 'larceny', 'lewd acts with a child', 'lying in wait', 'multiple murders',
                'murdered an officer in the line of duty', 'personal use of a handgun', 'poison', 'prior conviction',
                'prior offense', 'prior offenses', 'rape', 'robbery',
                'sexual assult', 'shooting at an occupied vehicle', 'sodomy', 'torture', 'unlawful possession of a firearm',]

In [26]:
updated_df = add_propensity(race_crime_cols, 'gender_women', 'gender', ohe_meta, full_meta, 'race_crime_match')


In [28]:
updated_df = add_propensity(state_race_crime_cols, 'gender_women', 'gender', ohe_meta, updated_df, 'state_race_crime_match')


In [27]:
updated_df.to_csv(save_path+ '/meta_with_psm_matches.csv')